# HW1 — Train a 2D Walking Robot with RL

Guided end-to-end version of the assignment. See `README.md` for the full
spec and grading. You edit three things:

- **Part A** — the observation: `planar_walker/env.py` → `_get_obs`
- **Part B** — the reward: `planar_walker/reward.py` → `compute_reward`, `is_healthy`
- **Part C** — the network: `planar_walker/networks.py` → `ActorCritic`

The PPO loop in `planar_walker/ppo.py` is given. Run cells top to bottom.

## 0. Setup
```bash
pip install -r requirements.txt
```
CPU only. Restart the kernel after editing any file in `planar_walker/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import torch

from planar_walker.config import EnvConfig, PPOConfig, RewardConfig
from planar_walker.env import PlanarWalkerEnv, QPOS_NAMES
from planar_walker.ppo import train, evaluate_policy, load_checkpoint

print('qpos layout:', QPOS_NAMES)

## 1. Sanity-check the environment
Random torques. The robot should flail and fall — that's expected.

In [ ]:
env = PlanarWalkerEnv()
obs, _ = env.reset(seed=0)
print('obs dim:', env.observation_space.shape, '  act dim:', env.action_space.shape)
print('control dt:', env.cfg.control_dt, 's   max episode steps:', env.cfg.max_episode_steps)

total = 0.0
for t in range(300):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    total += r
    if term or trunc:
        print(f'episode ended at t={t}, return so far {total:.1f}')
        break
env.close()

### Render a short clip
If offscreen rendering fails on Linux, see the rendering notes in `README.md`
(`MUJOCO_GL=egl`). You can skip this cell and still do the assignment.

In [ ]:
import imageio
from IPython.display import Video

env = PlanarWalkerEnv(render_mode='rgb_array')
obs, _ = env.reset(seed=1)
frames = []
for t in range(200):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    frames.append(env.render())
    if term or trunc:
        obs, _ = env.reset()
env.close()
imageio.mimsave('random.mp4', frames, fps=int(1/PlanarWalkerEnv().cfg.control_dt))
Video('random.mp4', embed=True, width=480)

## Part A — Observation design
Open `planar_walker/env.py` and implement `_get_obs`. It returns a 1-D
`np.float32` array built from `self.data.qpos` and `self.data.qvel`.

Think about:
- Absolute forward position `qpos[0]` (`rootx`) — helpful or harmful?
- Raw joint velocities — clip them? scale them?
- Would foot-contact flags or a gait phase help? (optional, advanced)

The cell below prints the pieces so you can decide.

In [ ]:
env = PlanarWalkerEnv()
env.reset(seed=0)
for _ in range(20):
    env.step(env.action_space.sample())
print('qpos:', dict(zip(QPOS_NAMES, np.round(env.data.qpos, 3))))
print('qvel:', np.round(env.data.qvel, 3))
print('current _get_obs() ->', env._get_obs().shape)
env.close()

## Part B — Reward design
Open `planar_walker/reward.py` and implement `compute_reward` and `is_healthy`.

A workable starting point (the reference solution):
$$ r = w_\text{fwd}\,\min(\dot x,\ v^*) \;+\; b_\text{alive}\,\mathbb{1}[\text{healthy}] \;-\; w_\text{ctrl}\lVert a\rVert^2 $$
with termination when the torso leaves the healthy height/pitch box.

Try to break it: what does the gait look like if you delete the control cost?
the alive bonus? the speed cap? Keep notes and screenshots for the report.

Quick numerical check of your reward at a standing pose:

In [ ]:
from planar_walker.reward import compute_reward, is_healthy
cfg = RewardConfig()
env = PlanarWalkerEnv(); env.reset(seed=0)
qp0 = env.data.qpos.copy()
env.step(np.zeros(6)); qp1 = env.data.qpos.copy(); qv1 = env.data.qvel.copy()
r, info = compute_reward(qp0, qp1, qv1, np.zeros(6), env.cfg.control_dt, cfg)
print('healthy at start:', is_healthy(qp0, cfg))
print('reward (zero torque, ~standing):', round(r, 4))
print('components:', {k: round(v,4) for k,v in info.items()})
env.close()

## Part C — Policy / Value network
Open `planar_walker/networks.py` and implement `ActorCritic.__init__` and
`_build_mlp`. A 2×64 tanh MLP with orthogonal init and a state-independent
`log_std` parameter is a solid default. `get_action_and_value` /
`act_deterministic` are already written against those attributes.

In [ ]:
from planar_walker.networks import ActorCritic
net = ActorCritic(obs_dim=17, act_dim=6, hidden=(64, 64))
print(net)
x = torch.zeros(4, 17)
a, logp, ent, v = net.get_action_and_value(x)
print('action', a.shape, 'logp', logp.shape, 'value', v.shape)
print('#params:', sum(p.numel() for p in net.parameters()))

## 2. Train
`train()` returns the checkpoint path. We collect the log rows for plotting.
Start with a short run to confirm everything is wired, then do the full run.

In [ ]:
rows = []
def logger(row):
    rows.append(row)
    if 'eval/return_mean' in row:
        print(f"  [eval] step={row['global_step']:>9,}  return={row['eval/return_mean']:8.1f}  "
              f"dist={row['eval/distance_mean']:6.2f} m  speed={row['eval/speed_mean']:.2f} m/s")
    elif row['update'] % 10 == 0:
        print(f"upd {row['update']:>4}  step {row['global_step']:>9,}  epRet {row['ep_return_mean']:8.1f}  "
              f"KL {row['approx_kl']:.3f}  std {row['policy_std']:.2f}")

env_cfg = EnvConfig()
ppo_cfg = PPOConfig(total_timesteps=300_000)   # smoke test; bump to 1_200_000 for the real run
ckpt = train(ppo_cfg, env_cfg, ckpt_path='checkpoints/policy.pt', log_fn=logger)
print('saved', ckpt)

### Learning curves

In [ ]:
tr = [r for r in rows if 'ep_return_mean' in r]
ev = [r for r in rows if 'eval/return_mean' in r]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([r['global_step'] for r in tr], [r['ep_return_mean'] for r in tr])
ax[0].set_title('training episode return'); ax[0].set_xlabel('env steps')
ax[1].plot([r['global_step'] for r in ev], [r['eval/return_mean'] for r in ev], 'o-', label='return')
ax[1].plot([r['global_step'] for r in ev], [r['eval/distance_mean'] for r in ev], 's-', label='distance [m]')
ax[1].set_title('greedy-policy evaluation'); ax[1].set_xlabel('env steps'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Watch the trained gait

In [ ]:
agent, obs_rms, env_cfg, _ = load_checkpoint('checkpoints/policy.pt')
env = PlanarWalkerEnv(cfg=env_cfg, render_mode='rgb_array')
obs, _ = env.reset(seed=123)
frames, ret = [], 0.0
done = False
while not done:
    o = obs_rms.normalize(obs[None], clip=env_cfg.obs_clip) if obs_rms is not None else obs[None]
    a = agent.act_deterministic(torch.as_tensor(o, dtype=torch.float32)).numpy()[0]
    obs, r, term, trunc, info = env.step(a)
    frames.append(env.render()); ret += r; done = term or trunc
env.close()
print(f'return {ret:.1f}   distance {info["x_position"]:.2f} m   steps {len(frames)}')
imageio.mimsave('walk.mp4', frames, fps=int(1/env_cfg.control_dt))
Video('walk.mp4', embed=True, width=480)

## 4. Evaluation table
Report this (mean ± std over ≥10 episodes) for your final policy.

In [ ]:
ev = evaluate_policy(agent, obs_rms, env_cfg, n_episodes=10)
for k, v in ev.items():
    print(f'{k:24s} {v:8.3f}')

## 5. Report
Assemble parts A–C, the ablation/sweep plots, the evaluation table, the gait
video, and the ½-page reflection. Export this notebook to PDF
(`File → Download as → PDF`) or write a separate document — either is fine.